# 🌐 Module 07: High-Throughput Network I/O with HTTPX

In modern async microservices and data engineering, you need to make thousands of external HTTP calls.
Using `requests` inside async code freezes the entire application. We use `httpx.AsyncClient`.

### What We Will Master:
1. **`httpx.AsyncClient` vs `requests`**.
2. **The #1 Mistake**: Creating an `AsyncClient` per request vs Reusing a shared client.
3. **Connection Pooling Tuning**: `httpx.Limits(max_connections, max_keepalive_connections)`.
4. **Streaming Large Downloads**: Consuming gigabytes without memory exhaustion.
5. **Resilient Retries with Exponential Backoff & Jitter**.

In [1]:
import asyncio
import time

# Production pattern: Reusable HTTP connection pool simulation
class MockAsyncHttpClient:
    def __init__(self, max_connections: int = 10):
        self.max_connections = max_connections
        self.reused_pool = 0

    async def get(self, url: str):
        self.reused_pool += 1
        await asyncio.sleep(0.2) # Network latency
        return {"url": url, "status": 200, "data": "ok"}

client = MockAsyncHttpClient()
start = time.perf_counter()
urls = [f"https://api.company.com/v1/resource/{i}" for i in range(1, 6)]

print("🚀 Reusing a single httpx.AsyncClient connection pool...")
responses = await asyncio.gather(*(client.get(u) for u in urls))
print(f"Fetched {len(responses)} endpoints in parallel in {time.perf_counter() - start:.2f}s!")
print(f"Keep-alive connections reused: {client.reused_pool}")

🚀 Reusing a single httpx.AsyncClient connection pool...
Fetched 5 endpoints in parallel in 0.22s!
Keep-alive connections reused: 5


---
## 2. Resilient Retries with Exponential Backoff & Jitter

When calling third-party services (OpenAI, Stripe, payment gateways), networks fail transiently.
Never retry immediately in a tight loop! Use exponential backoff ($2^n$) plus random jitter to avoid the **Thundering Herd problem**.

In [2]:
import random

attempt_counter = 0
async def flaky_payment_gateway():
    global attempt_counter
    attempt_counter += 1
    if attempt_counter < 3:
        raise ConnectionResetError("HTTP 503 Service Unavailable")
    return {"status": 200, "payload": "payment_confirmed"}

async def call_with_retry(max_attempts: int = 5, base_delay: float = 0.1):
    for attempt in range(1, max_attempts + 1):
        try:
            return await flaky_payment_gateway()
        except ConnectionResetError as e:
            if attempt == max_attempts:
                raise
            # Exponential backoff: base_delay * 2^(attempt-1) + jitter
            delay = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.05)
            print(f"Attempt {attempt} failed: {e}. Backing off for {delay:.2f}s...")
            await asyncio.sleep(delay)

result = await call_with_retry()
print("Attempt 3 succeeded:", result)
print("✅ Request succeeded after transient failures!")

Attempt 1 failed: HTTP 503 Service Unavailable. Backing off for 0.22s...
Attempt 2 failed: HTTP 503 Service Unavailable. Backing off for 0.43s...
Attempt 3 succeeded: {'status': 200, 'payload': 'payment_confirmed'}
✅ Request succeeded after transient failures!
